# Compare Analysis Roots

Overlays several analysis roots' results side by side. **Load-only** — it reads
each root's `live_summary.joblib` (written by `context.save_run_summary` in
`live_inference_epoched.ipynb`'s last cell), so there is no MNE / inference re-run.
An analysis root is any `SessionPaths`-shaped output directory (e.g. `data/sub_001`,
or a `debug_snapshots/<name>/` scenario) — not a separate profile registry.

Three comparisons:
1. **CV AUC** per decoder (the real performance metric).
2. **Headline metrics table** — avg peak AUC, diagonal-dominance, mean target diagonal.
3. **Live-inference P(t)** per decoder on its own marker (inference-time behavior).

Run `live_inference_epoched.ipynb` once per root first (`SOURCE = "fl"`, Run All)
so each root has a `live_summary.joblib`.

*Validation only — not part of the app.*


In [ ]:
from pathlib import Path
import joblib
import numpy as np
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt

# Locate repo root (dir containing src/backend/).
_s = Path().resolve()
REPO_ROOT = next((p for p in [_s, *_s.parents] if (p / "src" / "backend").is_dir()), None)
assert REPO_ROOT is not None, "Could not find src/backend/ above the CWD."

# Analysis roots to compare (each needs a live_summary.joblib from a prior FL run
# of live_inference_epoched.ipynb). Add more subjects/scenarios here as they exist.
ROOTS = [REPO_ROOT / "data" / "sub_001"]

summaries = {}
for root in ROOTS:
    name = root.name
    f = root / "live_summary.joblib"
    if not f.is_file():
        print(f"  [skip] {name}: no live_summary.joblib — run live_inference_epoched.ipynb (SOURCE='fl') for it")
        continue
    summaries[name] = joblib.load(f)
    s = summaries[name]
    print(f"  [ok]   {name}: {len(s['tasks'])} decoders | avg peak AUC {s['average_peak_auc']:.3f}")

assert summaries, "No summaries found — run live_inference_epoched.ipynb (SOURCE='fl') per root first."
loaded = list(summaries)
PROFILE_COLORS = {name: plt.cm.tab10(i % 10) for i, name in enumerate(loaded)}
TASKS = summaries[loaded[0]]["tasks"]   # reference decoder list (first loaded root)


def pos_markers(s, task):
    """A decoder's positive label(s). Newer summaries store the full group in
    `task_pos_markers`; older ones only have the single `task_pos_marker`."""
    pm = s.get("task_pos_markers")
    if pm and task in pm and pm[task]:
        return list(pm[task])
    m = s.get("task_pos_marker", {}).get(task)
    return [m] if m is not None else []


print("\ncomparing:", loaded)


---
## 1. CV AUC per decoder (overlay)

One panel per decoder; each profile's diagonal cross-validated AUC overlaid.
Peak AUC per profile in the legend. This is the headline comparison.

In [ ]:
ncols = 3
nrows = (len(TASKS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.2 * nrows), squeeze=False)
for idx, task in enumerate(TASKS):
    ax = axes[idx // ncols][idx % ncols]
    for name in loaded:
        s = summaries[name]
        if task not in s["diagonal_auc"]:
            continue
        ax.plot(s["auc_times"], s["diagonal_auc"][task], color=PROFILE_COLORS[name],
                lw=1.8, label=f"{name} ({s['peak_auc'][task]:.3f})")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    ax.axvline(0.0, color="k", ls=":", lw=0.8)
    ax.set(title=task, ylim=(0.4, 1.0), xlabel="time (s)", ylabel="CV AUC")
    ax.legend(fontsize=7, loc="upper left")
for j in range(len(TASKS), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
fig.suptitle("CV diagonal AUC — profile comparison", y=1.02)
plt.tight_layout(); plt.show()

---
## 2. Headline metrics

`diag_dom` = how many decoders whose own marker is the max in their row of the
per-trained-tp diagonal table. `mean_target_diag` = mean P(positive) of each
decoder on its own marker at its trained tp.

In [ ]:
def target_diag(s, task):
    """Mean P(positive) on this decoder's own marker(s) at its trained tp.
    For grouped decoders this averages over the positive group."""
    vals = [s["diag_at_tp"][task].get(m) for m in pos_markers(s, task)]
    vals = [v for v in vals if v is not None]
    return sum(vals) / len(vals) if vals else None


def dominance(s):
    """Decoders whose best positive marker is the row max at the trained tp."""
    n = 0
    for task in s["tasks"]:
        vals = {m: v for m, v in s["diag_at_tp"][task].items() if v is not None}
        tvals = [vals[m] for m in pos_markers(s, task) if m in vals]
        if tvals and max(tvals) == max(vals.values()):
            n += 1
    return n

w = 18
header = "profile".ljust(w) + "avg_peakAUC".rjust(13) + "diag_dom".rjust(10) + "mean_target_diag".rjust(18) + "sugg_tp".rjust(9)
print(header)
print("-" * len(header))
for name in loaded:
    s = summaries[name]
    targets = [target_diag(s, t) for t in s["tasks"]]
    targets = [v for v in targets if v is not None]
    mean_t = sum(targets) / len(targets) if targets else float("nan")
    print(name.ljust(w)
          + f"{s['average_peak_auc']:.3f}".rjust(13)
          + f"{dominance(s)}/{len(s['tasks'])}".rjust(10)
          + f"{mean_t:.3f}".rjust(18)
          + f"{s['suggested_timepoint']:.2f}".rjust(9))

# Per-decoder peak AUC, profiles side by side.
print()
pw = 12
print("decoder".ljust(22) + "".join(n.rjust(pw) for n in loaded))
print("-" * (22 + pw * len(loaded)))
for task in TASKS:
    row = task.ljust(22)
    for name in loaded:
        v = summaries[name]["peak_auc"].get(task)
        row += (f"{v:.3f}" if v is not None else "n/a").rjust(pw)
    print(row)

---
## 3. Live-inference P(t) per decoder on its own marker (overlay)

One panel per decoder, showing that decoder's mean P(t) (±SEM) on its **own**
positive marker, one line per profile. Dotted vertical = each profile's trained
tp. (Optimistic on the training recording — read alongside the AUC above.)

In [ ]:
ncols = 3
nrows = (len(TASKS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.2 * nrows), squeeze=False)
for idx, task in enumerate(TASKS):
    ax = axes[idx // ncols][idx % ncols]
    markers0 = pos_markers(summaries[loaded[0]], task)
    for name in loaded:
        s = summaries[name]
        ms = pos_markers(s, task)
        means = [s["pt_mean"].get(task, {}).get(m) for m in ms]
        means = [m for m in means if m is not None]
        if not means:
            continue
        mean = np.mean(means, axis=0)   # average over the positive group
        ax.plot(s["t_grid"], mean, color=PROFILE_COLORS[name], lw=1.8, label=name)
        if len(ms) == 1:                # SEM band only meaningful for a single marker
            sem = s["pt_sem"].get(task, {}).get(ms[0])
            if sem is not None:
                ax.fill_between(s["t_grid"], mean - sem, mean + sem,
                                color=PROFILE_COLORS[name], alpha=0.12)
        tp = s["tp_by_task"].get(task)
        if tp is not None:
            ax.axvline(tp, color=PROFILE_COLORS[name], ls=":", lw=0.8, alpha=0.6)
    ax.axhline(0.5, color="gray", lw=0.6)
    ax.axvline(0.0, color="k", ls=":", lw=0.8)
    ax.set(title=f"{task} — '{'/'.join(markers0)}'", ylim=(0, 1),
           xlabel="time from marker (s)", ylabel="P(positive)")
    ax.legend(fontsize=7, loc="upper right")
for j in range(len(TASKS), nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
fig.suptitle("Live inference P(t) on each decoder's own marker — profile comparison", y=1.02)
plt.tight_layout(); plt.show()